# Legal-LLM — Train on Colab

Run this when local training on your laptop isn't viable (thermal / VRAM / uptime).

**Prerequisites:**
1. Runtime → Change runtime type → GPU (T4 is fine; L4/A100 faster)
2. A Hugging Face token (https://huggingface.co/settings/tokens)
3. A Weights & Biases API key (https://wandb.ai/authorize)

**Pipeline:**
1. Mount Drive (for durable checkpoint storage — survives session restarts)
2. Clone the repo from GitHub
3. Upload `data/processed/` from your laptop (files.upload)
4. Install deps
5. Train with `configs/colab.yaml` (~1-2 hrs on T4)
6. Zip the best adapter, save to Drive, and download
7. At home: `python -m src.train.merge --adapter <downloaded_path> --out models/merged`

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (checkpoint survival)

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/legal-llm-checkpoints", exist_ok=True)
print("Drive checkpoint path: /content/drive/MyDrive/legal-llm-checkpoints")

## 3. Clone repo

Replace `YOUR_USER/legal-llm` with your actual GitHub repo path.

In [ ]:
GITHUB_REPO = "YOUR_USER/legal-llm"  # <-- edit this

!rm -rf /content/legal-llm
!git clone https://github.com/{GITHUB_REPO}.git /content/legal-llm
%cd /content/legal-llm

## 4. Upload your processed data

Run this cell, then select `data/processed/train.jsonl`, `val.jsonl`, `test.jsonl` from your laptop.

In [ ]:
import os

from google.colab import files

# Ensure we're inside the cloned repo (survives a runtime restart that skipped Cell 3)
repo_root = "/content/legal-llm"
assert os.path.isdir(repo_root), "Run Cell 3 (git clone) first."
os.chdir(repo_root)

os.makedirs("data/processed", exist_ok=True)
print("Select train.jsonl, val.jsonl, test.jsonl from your laptop (multi-select allowed):")
uploaded = files.upload()
for name in uploaded:
    dest = f"data/processed/{name}"
    os.rename(name, dest)
    print(f"Moved {name} -> {dest}")

!ls -la data/processed/

## 5. Install ML deps

Colab ships with torch + CUDA. We add unsloth and the rest of our stack.

In [ ]:
!pip install -q -U unsloth
!pip install -q -U \
    'transformers>=4.45' \
    'datasets>=3.0' \
    'peft>=0.14' \
    'trl>=0.12' \
    'bitsandbytes>=0.44' \
    'accelerate>=1.0' \
    'wandb>=0.18' \
    'pydantic>=2.9' \
    'pyyaml>=6.0'

## 6. Auth: W&B and Hugging Face

In [ ]:
import os
from getpass import getpass

os.environ["WANDB_API_KEY"] = getpass("W&B API key: ")
os.environ["HF_TOKEN"] = getpass("HF token (press Enter to skip): ")

## 7. Point checkpoint output at Drive (survives session restarts)

In [ ]:
import torch
import yaml

cfg_path = "configs/colab.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
cfg["training"]["output_dir"] = "/content/drive/MyDrive/legal-llm-checkpoints"

# T4 (compute 7.5) has no bfloat16 support; use fp16. A100/L4 (8.0+) can use bf16.
has_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
cfg["training"]["bf16"] = bool(has_bf16)
cfg["training"]["fp16"] = not has_bf16

with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("output_dir ->", cfg["training"]["output_dir"])
print(f"bf16={cfg['training']['bf16']}  fp16={cfg['training']['fp16']}")

## 8. Train

~1-2 hours on T4. W&B URL will print early in the output. If the Colab session disconnects, just re-run this cell — the auto-resume logic picks up the latest checkpoint from Drive.

In [ ]:
!python -m src.train.train --config configs/colab.yaml

## 9. Package the best adapter for download

After training finishes, this zips `best/` and saves it to your Drive + triggers a browser download.

In [ ]:
import shutil

from google.colab import files

src_dir = "/content/drive/MyDrive/legal-llm-checkpoints/best"
out_zip = "/content/drive/MyDrive/legal-llm-adapter.zip"
shutil.make_archive(out_zip.removesuffix(".zip"), "zip", src_dir)
print(f"Wrote {out_zip}")
files.download(out_zip)

## 10. Back on your laptop

```bash
cd ~/Adithya/legal-llm
mkdir -p models/checkpoints/best
unzip ~/Downloads/legal-llm-adapter.zip -d models/checkpoints/best/
make merge          # creates models/merged/
make eval           # 4-way benchmark (requires OPENAI_API_KEY)
```